# Model experiments and evaluation

This notebook is the technical companion to the browser demo. It documents the two prediction tasks, the bounded sampling design, the compared models, how the winning models were selected, and what the held-out scores do and do not mean.

It does not retrain models: the 642 MB dataset and model binaries are intentionally not part of the public presentation repository. The reported scores below come from the saved experiment summary used by the static demo. The code cells read those published metrics from `web/demo_bundle.json` so the numbers shown here stay aligned with the demo.

## Project question

A flow record is a compact summary of a connection, such as byte counts, packet counts, duration, protocol indicators, and flags. The model does not inspect packet payloads. We evaluated two targets:

- **Binary screening:** predict `Label`, where 0 is Benign and 1 is an attack.
- **Multiclass classification:** predict `Attack`, one of 15 named categories including Benign.

These are related but separate models. The binary model answers “benign or attack?” The multiclass model attempts the harder “which category?” question.


## Why the dataset was sampled

The local Parquet file has **17,129,715 rows and 43 columns** (41 numeric predictors plus the two targets) and is about **642 MB**. Loading every row and every feature into one in-memory DataFrame would be wasteful for this prototype.

The experiment scans the file in Arrow batches of 100,000 rows, reads only selected columns, converts retained predictors to `float32`, and keeps a bounded per-class reservoir sample. Reservoir sampling makes later rows eligible for selection too; the sample is not just the first rows in the file.

- Binary sample: **142,000 rows** — 125,000 Benign and 17,000 attacks, close to the source class ratio.
- Multiclass sample: **135,126 rows** — up to 12,000 per category; categories with fewer than 12,000 examples are retained in full.
- Each sampled dataset is split with seed **43** into 70% training, 15% validation, and 15% test. (The command-line default is 42; seed 43 reproduces the reports.)

A bounded sample limits memory use, but the complete source file is still scanned to determine class populations and fill the reservoirs.


## Feature handling

The model receives **36 numeric predictors**. Both `Label` and `Attack` are removed from the input matrix, regardless of which one is the target, to prevent the answer or its duplicate label from leaking into the predictors.

Five fields are excluded from the baseline feature set:

1. `FTP_COMMAND_RET_CODE`: constant zero in the full audit, so it cannot separate classes.
2. `L4_SRC_PORT` and `L4_DST_PORT`: omitted as a conservative choice against endpoint-specific shortcuts.
3. `SRC_TO_DST_SECOND_BYTES` and `DST_TO_SRC_SECOND_BYTES`: finite source values but extreme magnitudes beyond the `float32` range; their formulas and units were not verified.

These are choices for this dataset and baseline, not universal rules. Validation-only ablations tested the excluded ports and the extreme rates after `log1p`. A duration ablation also checked whether the selected binary result depended on duration. None of those ablations used the test set.

The audit found duration equals zero for about 95.68% of Benign rows and 12.75% of attack rows. That strong association could reflect how records were produced. The duration ablation produced the same displayed HGB validation F1, but this does not prove that duration is free of artifacts or leakage.


## Train, validation, and test have different jobs

The experiment first splits the fixed sample into stratified train, validation, and test partitions (70/15/15). Stratification preserves each target category's approximate sample share in each partition.

- **Training** fits model parameters.
- **Validation** compares candidate models and selects the candidate with the task's chosen score.
- The winner is **refit on training plus validation**.
- **Test** is then used for the final held-out report.

This separation reduces the risk of choosing a model based directly on its reported test score. The later source-prior multiclass comparison used the same seed and test records as the earlier balanced multiclass report. Its model choice came from validation, but this comparison is not an independent replication or a fresh external test.


## Candidate models and selection criteria

We compared three familiar model types:

| Candidate | Configuration | Purpose |
|---|---|---|
| Majority baseline | `DummyClassifier(strategy="most_frequent")` | Shows what a trivial majority-only rule achieves. It predicts Benign for the binary task and has attack recall/F1 of zero. |
| Logistic regression | `StandardScaler` + `LogisticRegression(max_iter=1000, tol=0.001)` | A linear benchmark. Scaling keeps differently measured numeric features from dominating only because of their units. |
| Histogram gradient boosting | `HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=31, learning_rate=0.1, l2_regularization=1.0)` | An ensemble of decision trees that can learn nonlinear thresholds and feature interactions. |

Model selection used source-frequency-weighted **attack F1** for binary classification and source-frequency-weighted **macro-F1** for multiclass classification. The validation scores below are selection results, not the final test scores.


In [ ]:
# Validation scores recorded in the experiment reports
validation = {
    "Binary attack F1": {
        "Majority baseline": 0.0000,
        "Logistic regression": 0.8529,
        "Histogram gradient boosting": 0.9766,
    },
    "Multiclass macro-F1 (balanced weights)": {
        "Majority baseline": 0.0625,
        "Logistic regression": 0.5680,
        "Histogram gradient boosting": 0.6907,
    },
    "Multiclass macro-F1 (source-prior weights)": {
        "Majority baseline": 0.0625,
        "Logistic regression": 0.6118,
        "Histogram gradient boosting": 0.7586,
    },
}
for task, scores in validation.items():
    print(task)
    for model, score in scores.items():
        print(f"  {model:32} {score:.4f}")


## Imbalance and source-prior weighting

The complete dataset contains 15,101,685 Benign records and 2,028,030 attack records. Multiclass category sizes vary sharply: SQL Injection has 432 source rows and Brute Force -XSS has 927, while some categories have over one million. Capping every category at 12,000 makes the training sample much more balanced than the source file.

For a sampled row of class *c*, the population adjustment is:

`row weight = source rows in class c / sampled rows in class c`

Those weights are normalized to mean 1 when used for source-prior model fitting. Evaluation metrics are separately weighted by class population counts to estimate scores under the local file's class proportions. These weights correct for the sampling design; they do not guarantee that the source data represents a future network.

Two multiclass fits were compared:

- **Balanced training weights** give rarer classes more influence. That can help recall, but can also produce many false-positive assignments to rare classes.
- **Source-prior weights** restore the source class proportions during fitting. This improved aggregate source-weighted scores and reduced some rare-class false positives, while rare-class recall fell.


In [ ]:
# Example of how source-prior weighting changes the capped sample
source_counts = {"Benign": 15_101_685, "SQL Injection": 432}
sample_counts = {"Benign": 12_000, "SQL Injection": 432}
for category in source_counts:
    weight = source_counts[category] / sample_counts[category]
    print(f"{category:15} raw class weight: {weight:,.2f}")

print("Weights are normalized to mean 1 before model fitting.")


## Reading the binary test result

The selected binary candidate was histogram gradient boosting. These are source-weighted metrics on the held-out sample:

- **Precision** asks: among flows flagged as attacks, what share were attacks?
- **Recall** asks: among actual attacks, what share did the model find?
- **F1** is the harmonic mean of precision and recall; it is high only when both are reasonably strong.
- **ROC-AUC** measures ranking across score thresholds. **PR-AUC** summarizes precision and recall and is useful when the positive class is less common.
- **False-positive rate** asks what share of benign records were flagged as attacks.

The displayed confusion matrix below contains raw sample counts, not weighted population estimates. Keep those two views distinct when presenting.


In [ ]:
# Complete held-out metrics from the experiment report. The public JSON bundle
# contains a smaller subset for the website cards; we check the shared values.
binary_report = {
    "accuracy": 0.9951649,
    "attack_precision": 0.9954865,
    "attack_recall": 0.9635294,
    "attack_f1": 0.9792473,
    "roc_auc": 0.9893172,
    "pr_auc": 0.9775778,
    "false_positive_rate": 0.0005867,
}
multiclass_report = {
    "accuracy": 0.9942318,
    "macro_f1": 0.7886690,
    "weighted_f1": 0.9931520,
}
assert abs(binary_report["attack_f1"] - binary["attack_f1"]) < 1e-5
assert abs(binary_report["attack_precision"] - binary["attack_precision"]) < 1e-5
assert abs(binary_report["attack_recall"] - binary["attack_recall"]) < 1e-5
assert abs(multiclass_report["macro_f1"] - multiclass["macro_f1"]) < 1e-5
assert abs(multiclass_report["accuracy"] - multiclass["accuracy"]) < 1e-5

print("Binary selected-model test metrics (source weighted)")
for key, value in binary_report.items():
    print(f"{key:24} {value:.6f}")
print("\nMulticlass source-prior test metrics (source weighted)")
for key, value in multiclass_report.items():
    print(f"{key:24} {value:.6f}")


In [ ]:
# Raw sampled binary confusion matrix: actual rows by predicted columns
confusion = {
    "Actual Benign": {"Predicted Benign": 18_739, "Predicted Attack": 11},
    "Actual Attack": {"Predicted Benign": 93, "Predicted Attack": 2_457},
}
for actual, predictions in confusion.items():
    print(actual, predictions)

print("\nAt this sample threshold: 11 false alarms and 93 missed attacks.")
print("The deployment threshold was not tuned to an operational security team's costs.")


The source-weighted binary test metrics were **0.9952 accuracy, 0.9955 attack precision, 0.9635 attack recall, and 0.9792 attack F1**. Weighted false-positive rate was about **0.000587**. On the unweighted sampled confusion matrix, 11 benign records were flagged and 93 attacks were missed. This is strong performance on this dataset's held-out sample, not proof of performance on another network.

A security team may prefer a different threshold depending on whether false alarms or missed attacks are more costly. The project did not tune a production threshold or deploy an alerting service.


## Multiclass results and the rare-class trade-off

The source-prior model had **0.7887 macro-F1**, **0.9942 accuracy**, and **0.9932 weighted-F1** on the weighted held-out sample. The earlier balanced model had macro-F1 0.6915, accuracy 0.5737, and weighted-F1 0.7180 on the same sampled split.

The high source-prior accuracy is dominated by common categories. Macro-F1 calculates F1 for each class and averages those scores equally, so it gives small categories a voice. A good aggregate accuracy can therefore coexist with poor detection of a rare attack.

The rare-class results below are from the source-prior run. “Support” is the raw number of sampled test examples, not the weighted population estimate.


In [ ]:
rare_classes = [
    ("Brute Force -Web", 0.9250, 0.1153, 0.2050, 321),
    ("Brute Force -XSS", 1.0000, 0.1223, 0.2179, 139),
    ("Infilteration", 0.6881, 0.3211, 0.4379, 1800),
    ("SQL Injection", 0.0000, 0.0000, 0.0000, 65),
]
print(f"{'Category':22} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
for name, precision, recall, f1, support in rare_classes:
    print(f"{name:22} {precision:10.4f} {recall:10.4f} {f1:10.4f} {support:10,d}")


The model missed all **65** sampled SQL Injection test examples. The source category contains only 432 records total, so the test support is small and this result has uncertainty. It is still a clear failure on the evaluated sample and should be stated directly.

Balanced weighting gave much higher recall for some rare categories but extremely low precision. For example, balanced Infilteration precision was about 0.015 with recall about 0.893. Source-prior weighting raised precision to about 0.688 while recall fell to about 0.321. Which trade-off is preferable depends on the intended use and its error costs.


## Feature ablations

Ablation experiments change one feature choice at a time. These are **binary validation-only** attack-F1 scores; they are not test results:

| Setup | Logistic regression | Histogram gradient boosting |
|---|---:|---:|
| Final feature set | 0.8529 | 0.9766 |
| Drop flow duration | 0.8399 | 0.9766 |
| Add two extreme rate features after `log1p` | 0.8513 | 0.9766 |
| Include source and destination ports | 0.8624 | 0.9764 |

At the shown precision, adding ports or transformed rates did not improve HGB validation F1, and removing duration left its displayed score unchanged. These experiments do not establish causal feature importance, robust generalization, or absence of data artifacts. Do not compare these validation scores directly with the held-out test scores above.


## How the browser demo works

The public page is a static representation, not a live inference service. Four selected example flows were run through the locally saved binary and multiclass models. A bundle-building step wrote the example descriptions, predictions, scores, and test metrics to `web/demo_bundle.json`; `web/index.html` reads that JSON and renders the experience.

The browser does not load the training dataset or model binaries, accept arbitrary user-provided feature vectors, or capture network traffic. The example score is not a calibrated probability. The demo is useful for explaining the model's outputs and showing a known failure, especially the SQL Injection miss.

A future NetFlow/IPFIX extension would need to map incoming records to the same 36 predictors, handle missing/invalid values, and be validated with labels from the intended network before anyone relied on it.


## Limits to be ready to explain

- The model was evaluated on one labeled dataset and a bounded sample from it, not an external network.
- The split is random and stratified by row. Exact duplicates, capture-level dependence, and time-based generalization were not fully tested.
- The later source-prior multiclass result uses the same sampled test rows as the earlier balanced report; it is not an independent replication.
- Rare-class results have small supports. SQL Injection has only 65 sampled test examples and all were missed by the source-prior model.
- The binary threshold was not optimized for operational costs.
- The demo uses precomputed examples and outputs; it is not live inference.
- Model scores have not been calibrated as probabilities.

## Short technical answer: what is the contribution?

The project contribution is a flow-based evaluation pipeline on the local 43-column dataset: EDA and quality checks, a bounded sampling strategy, binary and 15-category tasks, model comparison against a majority baseline and logistic regression, a class-weighting experiment, and a public demonstration that includes a failure case. It does not claim a new algorithm or a reproduction of the referenced CNN paper.
